# 🧩 OOP in Python — Part 2: Practice & Deeper Concepts

This notebook builds two full mini-projects (a simple ATM, then a private/secure ATM) and then digs into the deeper mechanics of Python OOP: real privacy conventions, getters/setters, reference semantics, static vs instance variables, and object relationships (Aggregation, Inheritance).

## 🎯 Learning Goal

By the end of this notebook you should understand:
- How to build a stateful, menu-driven class (the ATM) using instance attributes and a loop
- Why Python has **no true private variables** — only naming *conventions* (`_x`) and **name mangling** (`__x`)
- The difference between a traditional getter/setter and the more Pythonic `@property` approach
- Why passing an object into a function passes a **reference**, and how that affects mutation
- The difference between a **class variable** (shared) and an **instance variable** (per-object)
- What **Aggregation** ("Has-A") and **Inheritance** ("Is-A") relationships look like in code

## 🤔 What's Different in Part 2? (Real-life analogy)

Part 1 taught you what a class *is*. Part 2 is about what happens when objects **talk to each other** and **live inside a running program**.

Think of a bank vault 🏦: the outer door (public methods) is what customers use. The inner mechanism (private attributes like `__pin`, `__balance`) is deliberately hidden — Python doesn't *physically* lock it (unlike Java's `private`), it just renames the lock (**name mangling**) and trusts you not to pick it. That's the "consenting adults" philosophy you'll see quoted below.

### 🧠 Key Idea

- Python has **no enforced privacy**. `_x` is a *convention* ("please don't touch this from outside"); `__x` triggers **name mangling** (`_ClassName__x`), which mostly just makes accidental access harder, not impossible.
- `@property` lets you call a method like an attribute (`obj.balance` instead of `obj.get_balance()`) while still running validation code behind the scenes.
- Objects are passed to functions **by reference** — mutating the object *inside* the function affects the original object outside it too. Reassigning the parameter name does not.
- A **class variable** is shared by every instance (great for counters); an **instance variable** belongs to one object only.
- **Aggregation** ("Has-A"): one class holds another class as an attribute (e.g. `Customer` has an `Address`). **Inheritance** ("Is-A"): one class extends another and gets all its methods for free (e.g. `Student` is a `User`).

### 📚 Important Terms

| Term | Simple Meaning | Example |
|---|---|---|
| `_variable` | Convention meaning "internal use only" (not enforced) | `self._balance` |
| `__variable` | Triggers name mangling to `_ClassName__variable` | `self.__pin` |
| Name Mangling | Python renaming `__x` internally to make outside access awkward | `self.__vault_code` → `_BankAccount__vault_code` |
| `@property` | Decorator that turns a method into a read-like attribute | `@property def balance(self):` |
| `@x.setter` | Decorator that lets `obj.x = value` run custom validation | `@balance.setter` |
| Reference | A variable that points to an object in memory, not a copy of it | `cust = BankAccount(...)` |
| `id()` | Returns an object's memory address (identity) | `id(cust)` |
| Class Variable | Shared across all instances of a class | `Customer.counter` |
| Instance Variable | Unique to one object | `self.name` |
| Aggregation | "Has-A" relationship — one class contains another as data | `Customer.address` |
| Inheritance | "Is-A" relationship — a subclass reuses a parent class's methods | `class Student(User):` |

## 1. 🏧 Building an ATM (Simple Version)

A menu-driven ATM machine that runs in a loop, taking input until the user chooses to exit. This shows a class that manages **state** (`pin`, `balance`) across many method calls.

In [12]:
class AtmMachine:
    
    def __init__(self):
        self.pin = ""
        self.balance = 0
        self.menu()
    
    def menu(self):
        while True:
            choice = input(
                """
            Please choose any option
            1. Press 0 to add balance
            2. Press 1 to create pin
            3. Press 2 to change pin
            4. Press 3 to check balance
            5. Press 4 to withdraw
            6. Press any other key to exit
                """
                )
            
            if choice == "0":
                self.add_balance()
            elif choice == "1":
                self.create_pin()
            elif choice == "2":
                self.change_pin()
            elif choice == "3":
                self.check_balance()
            elif choice == "4":
                self.withdraw()
                pass
            else:
                print("Thank you for using the ATM. Goodbye!")
                break 
        
        
    def create_pin(self):
        self.user_pin = input("Enter your pin: ")
        self.pin = self.user_pin
        print("PIN created successfully!")
        
    def add_balance(self):
        amount = int(input("Enter the amount you want to add: "))
        self.balance += amount
        print(f"{amount} is added to your account")
        print("Current Balance: ", self.balance)
    
    def check_balance(self):
        print("Your Current balance is:", self.balance)
    
    def change_pin(self):
        old_pin = input("Enter your current pin: ")
        if old_pin == self.pin:
            self.pin = input("Enter your new pin: ")
            print("PIN changed successfully!")
        else:
            print("Incorrect PIN!")
            self.change_pin()
        
    def withdraw(self):
        amount = int(input("Enter amount to withdraw: "))
        if amount <= self.balance:
            self.balance -= amount
            print(f"{amount} withdrawn successfully.")
        else:
            print("Insufficient balance!")
        

**Note:** `self.menu()` is called at the very end of `__init__`, so simply creating the object (`obj = AtmMachine()`) immediately starts the interactive menu loop — there's no separate "start" method to call.

**Gotcha:** Nothing stops you from calling `withdraw()` or `check_balance()` before ever calling `create_pin()` — `self.pin` starts as `""` and `self.balance` starts as `0`, so those methods will just silently operate on the defaults rather than warning you no PIN has been set yet.

In [13]:
obj = AtmMachine()

1250 is added to your account
Current Balance:  1250
PIN created successfully!
Incorrect PIN!
Incorrect PIN!
PIN changed successfully!
Your Current balance is: 1250
250 withdrawn successfully.
Your Current balance is: 1000
Thank you for using the ATM. Goodbye!


**Note:** This cell is **interactive** — running it will prompt you for input in the notebook. The pattern below in Section 2 fixes the biggest weakness here: **anyone can call `add_balance()`, `withdraw()`, or `check_balance()` without ever entering a PIN**, because none of those methods check `self.pin`. Section 2 locks that down.

## 2. 🔢 A `Fraction` Class — Operator Overloading

Python lets you redefine what `+`, `-`, `*`, `/` mean for your own objects by implementing "dunder" (double-underscore) methods like `__add__`, `__sub__`, `__mul__`, `__truediv__`, and `__str__`.

In [30]:
class Fraction:
    
    def __init__(self, n, d):
        self.num = n
        self.den = d

        
    def __str__(self):
        # return "Hello"
        return "{}/{}".format(self.num, self.den)
    
    def __add__(self, other):
        temp_num = self.num * other.den + other.num * self.den
        temp_den = self.den * other.den
        
        return "{}/{}".format(temp_num, temp_den)

    def __sub__(self, other):
        temp_num = self.num * other.den - other.num * self.den
        temp_den = self.den * other.den
        
        return "{}/{}".format(temp_num, temp_den)
    
    def __mul__(self, other):
        temp_num = self.num * other.num
        temp_den = self.den * other.den
        
        return "{}/{}".format(temp_num, temp_den)
    
    def __truediv__(self, other):
        temp_num = self.num * other.den
        temp_den = self.den * other.num
        
        return "{}/{}".format(temp_num, temp_den)
        

x = Fraction(3, 4)
print(x)

y = Fraction(5, 6)
print(y)

print(x+y)
print(x-y)
print(x*y)
print(x/y)

3/4
5/6
38/24
-2/24
15/24
18/20


**Note:** `__str__` controls what `print(x)` displays — without it, `print(x)` would show something unhelpful like `<__main__.Fraction object at 0x...>`.

**Gotcha:** `__add__`, `__sub__`, `__mul__`, and `__truediv__` all `return` a **plain string** (via `"{}/{}".format(...)`), not a new `Fraction` object. That means `x + y` gives you back a string, and you can print it fine, but you **cannot chain operations** — `(x + y) + z` would fail, because a string doesn't have `__add__` defined the way `Fraction` does (string `+` would just concatenate text instead). Also, the results aren't simplified — `x*y` above gives `15/24` instead of the reduced `5/8`. A more complete version would `return Fraction(temp_num, temp_den)` and reduce using `math.gcd`. See the extra example below for a fixed version.

### Extra Example — Fixing `Fraction` to Support Chaining

Here's the same class, but `__add__` now returns an actual `Fraction` object (and reduces it using `math.gcd`), so operations can be chained: `(x + y) + z` works correctly.

In [ ]:
import math

class FractionV2:
    
    def __init__(self, n, d):
        self.num = n
        self.den = d
    
    def _reduce(self, n, d):
        g = math.gcd(n, d)
        return FractionV2(n // g, d // g)
    
    def __str__(self):
        return "{}/{}".format(self.num, self.den)
    
    def __add__(self, other):
        temp_num = self.num * other.den + other.num * self.den
        temp_den = self.den * other.den
        return self._reduce(temp_num, temp_den)
    
    def __mul__(self, other):
        temp_num = self.num * other.num
        temp_den = self.den * other.den
        return self._reduce(temp_num, temp_den)

x = FractionV2(1, 4)
y = FractionV2(1, 4)
z = FractionV2(1, 2)

print(x + y)        # 1/2  (reduced from 2/4)
print((x + y) + z)  # 1/1  (chaining works because __add__ returns a FractionV2)
print(x * y)        # 1/16


**Note:** Compare this to the original `Fraction` above — because `__add__` now returns `FractionV2` (via `self._reduce(...)`), the result can be added *again* in `(x + y) + z`. This is the standard fix for the "operator overloading returns a plain value instead of a new object" gotcha.

# Encapsulation
### In Python, there is a famous saying among developers: "We are all consenting adults here." This philosophy is the core reason why Python does not enforce true privacy. Unlike languages like Java or C++, which have strict access modifiers (private, protected, public) enforced by the compiler, Python relies on convention, trust, and transparency.

## 3. 🔒 Encapsulation with Name Mangling — a Private ATM

Prefixing an attribute or method with `__` (double underscore) triggers **name mangling**: Python internally renames `self.__pin` to `self._ClassName__pin`. This doesn't make it truly private, but it does prevent *accidental* access and name clashes in subclasses.

In [ ]:
## Encapsulation Use __ in front of data properties or method properties to hide and restrict the user to access.

class AtmMachine:
    
    def __init__(self):
        self.__pin = ""
        self.__balance = 0
        self.__menu()
    
    def __menu(self):
        while True:
            choice = input(
                """
            Please choose any option
            1. Press 1 to create pin
            2. Press 2 to add balance
            3. Press 3 to withdraw
            4. Press 4 to check balance
            5. Press 5 to change pin
            6. Press any other key to exit
                """
                )
          
            if choice == "1":  #  Press 1 to create pin
                self.create_pin()
            elif choice == "2":  #  Press 2 to add balance
                 self.deposit()
            elif choice == "3":  #  Press 3 to withdraw
                self.withdraw()
            elif choice == "4": #  Press 4 to check balance
                self.check_balance()
            elif choice == "5": # Press 5 to change pin
                self.change_pin()
            else:
                print("Thank you for using the ATM. Goodbye!")
                break 
        
        
    def create_pin(self):
        self.user_pin = input("Enter your pin: ")
        self.__pin = self.user_pin
        print("PIN created successfully!")
        
    def deposit(self):
        temp = input("Enter your Current pin to add Balance: ")
        if temp == self.__pin:
            amount = int(input("Enter the amount you want to add: "))
            self.__balance += amount
            print(f"{amount} is added to your account")
            print("Current Balance: ", self.__balance)
        else:
            print("Invalid Pin")
            
    def withdraw(self):
        temp = input("Enter your Current pin to add Withdraw: ")
        if temp == self.__pin:
            amount = int(input("Enter amount to withdraw: "))
            if amount <= self.__balance:
                self.__balance -= amount
                print(f"{amount} withdrawn successfully.")
            else:
                print("Insufficient balance!")
        else:
             print("Incorrect PIN!")
        
    
    def check_balance(self):
        temp = input("Enter your Current pin to check Balance: ")
        if temp == self.__pin:
            print("Your Current balance is:", self.__balance)
        else:
            print("Invalid Pin")
    
    def change_pin(self):
        old_pin = input("Enter your current pin: ")
        if old_pin == self.__pin:
            self.__pin = input("Enter your new pin: ")
            print("PIN changed successfully!")
        else:
            print("Incorrect PIN!")
            self.change_pin()
        
   
    
# hdfc = AtmMachine()
sbi = AtmMachine()  


**Bug fixed:** the printed menu text still described the *old* option numbering from the Section 1 ATM ("Press 0 to add balance", "Press 1 to create pin", ...), but the actual `if/elif` checks in this class use a completely different mapping (`"1"` = create pin, `"2"` = deposit, etc.). The menu text has been corrected to match what the code actually does — this was a real "the comment lies about the code" bug that would have confused anyone using it.

**Note:** `self.__menu` (with double underscore) is itself name-mangled too — it becomes `self._AtmMachine__menu` internally. This is why the interactive demo below still works fine from *inside* the class, but you could not call `sbi.__menu()` from outside without also mangling the name yourself.

**Gotcha:** `self.user_pin = input(...)` inside `create_pin` creates a *new, non-mangled* public attribute (`user_pin`) as well as setting `self.__pin`. That's an accidental leftover — it means the pin is technically still readable from outside via `sbi.user_pin`, defeating some of the purpose of using `__pin` in the first place. A stricter version would just do `self.__pin = input("Enter your pin: ")` directly.

### 3.1 The "Single Underscore" Convention (`_variable`)

A single leading underscore is a **convention only** — it signals "internal use, please don't touch" but Python does nothing to stop you.

In [14]:
class BankAccount:
    def __init__(self):
        self._balance = 1000  # Intended to be private
        
account = BankAccount()
print(account._balance)  # Works perfectly fine! 1000 | It acts like a "Keep Out" sign on an unlocked door. You can walk right in if you choose to ignore the sign.
        

1000


**Note:** `account._balance` works with zero errors — Python enforces *nothing* here. The underscore is purely a social contract between developers, like a "Keep Out" sign on an unlocked door.

### 3.2 Name Mangling (`__variable`)

A double leading underscore does something real: Python **renames** the attribute internally to `_ClassName__attribute`. This is called **name mangling**.

In [15]:
class BankAccount:
    def __init__(self):
        self.__vault_code = 9999  # Sounds private, right?

account = BankAccount()

print(account.__vault_code) 

# Accessing the mangled name directly:
# print(account._BankAccount__vault_code)  # Works perfectly! 9999

AttributeError: 'BankAccount' object has no attribute '__vault_code'

**Note:** This cell **intentionally errors** — `account.__vault_code` fails with `AttributeError` because Python silently renamed the attribute to `_BankAccount__vault_code` behind the scenes. There is no attribute literally called `__vault_code` on the object; name mangling happened at class-definition time.

In [16]:
class BankAccount:
    def __init__(self):
        self.__vault_code = 9999  # Sounds private, right?

account = BankAccount()

# Accessing the mangled name directly:
print(account._BankAccount__vault_code)  # Works perfectly! 9999

9999


**Note:** Accessing the mangled name directly (`account._BankAccount__vault_code`) works and prints `9999`. This proves name mangling is **obfuscation, not real security** — the data is still there and still reachable if you know (or guess) the mangled name.

### 3.3 Full Reflection and Introspection

Using built-in functions like `dir()`, `vars()`, or `getattr()`, you can peek inside *any* object — including its mangled attributes — and read or modify them dynamically.

In [ ]:
class BankAccount:
    def __init__(self):
        self.owner = "Alice"
        self._balance = 1000
        self.__vault_code = 9999

account = BankAccount()

# Print the dictionary of attributes
print(vars(account))

# Pass the object inside dir()
print(dir(account))

# 1. Accessing a standard variable
print(getattr(account, "owner"))          # Output: Alice

# 2. Accessing a semi-private variable
print(getattr(account, "_balance"))       # Output: 1000

# 3. Accessing a mangled private variable
print(getattr(account, "_BankAccount__vault_code"))  # Output: 9999

# 4. Using a default value to prevent crashing if the attribute doesn't exist
print(getattr(account, "routing_number", "Not Found")) # Output: Not Found

{'owner': 'Alice', '_balance': 1000, '_BankAccount__vault_code': 9999}
['_BankAccount__vault_code', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_balance', 'owner']
Alice
1000
9999
Not Found


**Note:** `vars(account)` returns the object's actual `__dict__` — notice the key is `_BankAccount__vault_code`, not `__vault_code`, confirming the mangling happened at assignment time (`self.__vault_code = 9999` inside `__init__`), not just at read time.

**Note:** `getattr(obj, name, default)` is the safe way to read a possibly-missing attribute without a `try/except` — it returns `"Not Found"` here instead of raising `AttributeError` for `routing_number`, which was never set.

## 4. 🔑 Getter and Setter in Python

Since Python doesn't enforce privacy, getters/setters exist to add **validation** when reading or writing a value — not to "protect" it from access.

In [ ]:
# 1. The Traditional Way (Methods/Functions): If you prefer explicit function calls (like get_x() and set_x()), you can write standard methods.

class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        self._balance = balance

    # Getter Method
    def get_balance(self):
        return self._balance

    # Setter Method
    def set_balance(self, new_amount):
        if new_amount >= 0:
            self._balance = new_amount
        else:
            print("Invalid amount")

# --- How to use it ---
account = BankAccount("Bob", 2000)

# Using the Getter method
print(account.get_balance())  # Output: 2000

# Using the Setter method
account.set_balance(2500)
print(account.get_balance())  # Output: 2500

2000
2500


**Note:** This is the traditional, Java-style approach: explicit `get_balance()` / `set_balance()` method calls. It works, but it's not the idiomatic way to do this in Python.

In [ ]:
# 2. The Pythonic Way (Recommended: @property)

class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        self._balance = balance  # Internal variable

    # The Getter
    @property
    def balance(self):
        print("Fetching the balance safely...")
        return self._balance

   # The Setter
    @balance.setter
    def balance(self, new_amount):
        if new_amount < 0:
            print("Error: Balance cannot be negative!")
        else:
            print("Updating balance...") 
            self._balance = new_amount

# --- How to use it ---
account = BankAccount("Alice", 1000)

# Using the Getter (Notice: No parentheses needed!)
print(account.balance)      # Output: Fetching the balance safely... -> 1000

# Using the Setter (Notice: You use the assignment '=' operator)
account.balance = 1500      # Output: Updating balance...
account.balance = -500      # Output: Error: Balance cannot be negative!

print(account.balance)      # Latets Balance

Fetching the balance safely...
1000
Updating balance...
Error: Balance cannot be negative!
Fetching the balance safely...
1500


**Note:** With `@property`, `account.balance` (no parentheses!) calls the getter method behind the scenes, and `account.balance = 1500` (plain assignment) calls the setter — but the setter's validation (`if new_amount < 0`) still runs. This is the **Pythonic** way to add validation while keeping the clean "attribute access" syntax.

**Gotcha:** Once you define `balance` as a `@property`, you can no longer do `self.balance = ...` inside `__init__` without going through the setter too — that's actually a *feature* here (the setter's validation applies even during construction), but it can be a surprise if you expected `__init__` to bypass validation.

## 5. 🔗 Reference Variables in Python

When you pass an object to a function, you're passing a **reference** (the memory address), not a copy. This means the function can mutate the original object.

In [44]:
class BankAccount:
    def __init__(self, name):
        self.name = name

def greet(customer_obj):
    print(id(customer_obj))

cust = BankAccount("Ankita")
print(id(cust))
greet(cust)

4475465168
4475465168


**Note:** `id(cust)` printed twice (once outside, once inside `greet`) shows the **same** number — `cust` and `customer_obj` are two different names pointing at the *same* object in memory.

In [47]:
class BankAccount:
    def __init__(self, name):
        self.name = name

def greet(customer_obj):
    customer_obj.name = "Nitish"
    print(customer_obj.name)

cust = BankAccount("Ankita")
greet(cust)

print(cust.name)

Nitish
Nitish


**Note:** `greet` sets `customer_obj.name = "Nitish"` — this **mutates the object's attribute**, not the reference itself. Since `cust` and `customer_obj` point to the same object, `cust.name` is now `"Nitish"` too, even after the function returns.

In [ ]:
class BankAccount:
    def __init__(self, name):
        self.name = name

def greet(customer_obj):
    print(id(customer_obj))
    customer_obj.name = "Nitish"
    print(customer_obj.name)
    print(id(customer_obj))

cust = BankAccount("Ankita")
greet(cust)

print(cust.name)

## class ke object are also mutable like list dictionary and sets



4477005584
Nitish
4477005584
Nitish


**Note:** This combines the two ideas above — the `id()` stays identical before and after the mutation (same object, same address), but `.name` has changed. Mutating an attribute never changes an object's identity.

In [42]:
class BankAccount:
    def __init__(self, name, gender):
        self.name = name
        self.gender = gender

def greet(customer_obj):
    if customer_obj.gender == "Male":
        print("Hello,", customer_obj.name, "Sir")
    elif customer_obj.gender == "Female":
        print("Hello,", customer_obj.name, "Mam")
    else:
         print("Wrong Gender Input")
         
    cust2 = BankAccount("Nitish", "Male")
    return cust2

cust = BankAccount("Ankita", "Female")
new_cust = greet(cust)
print(new_cust.name)

Hello, Ankita Mam
Nitish


**Note:** This one is a common source of confusion — inside `greet`, `customer_obj` refers to `cust` ("Ankita"), and the printed greeting correctly uses her data. But then the function creates a **brand-new** `BankAccount` object (`cust2`, "Nitish") and returns *that* instead of `cust`. So `new_cust.name` is `"Nitish"`, not `"Ankita"` — the original `cust` object was never modified; a completely different object was returned. This demonstrates that **reassigning inside a function (or returning a new object) does not affect the caller's original variable**, unlike mutating an existing object's attribute (as in the two cells above).

**Note:** Just like lists, dicts, and sets, class objects in Python are **mutable** — the two cells above are proof of this.

### 5.1 Comparing to Lists and Tuples (Mutable vs Immutable)

Class objects behave like lists (mutable, mutation is visible to the caller) — but not like tuples (immutable, so "changing" one actually creates a new object).

In [50]:
## List example of pass by reference

def change(L):
    print(id(L))
    L.append(5)
    print(id(L))
    

L1 = [1, 2, 3, 4]
print(id(L1))
print(L1)

change(L1)
print(L1)

print("---------")
L2 = [1, 2, 3, 4]
print(id(L2))
print(L2)

change(L2[:]) ## try to use clonning to avoid data rewrite in inside of method in python
print(L2)




4476070016
[1, 2, 3, 4]
4476070016
4476070016
[1, 2, 3, 4, 5]
---------
4476064704
[1, 2, 3, 4]
4475679680
4475679680
[1, 2, 3, 4]


**Note:** `change(L1)` mutates the list **in place** (`L.append(5)`) — the `id()` doesn't change, and `L1` outside the function now includes the `5` too. But `change(L2[:])` passes a **slice-copy** of `L2` — a brand-new list object — so appending to it inside the function has zero effect on the original `L2`. Slicing (`L2[:]`) is a classic trick to avoid unwanted mutation.

In [ ]:
## Tuple example of pass by reference

def change(L):
    print(id(L))
    L = L + (5, 6) ## here id will chnage but change will not happen in tupple
    print(id(L))
    

L1 = (1, 2, 3, 4)
print(id(L1))
print(L1)

change(L1)
print(L1)

4468905744
(1, 2, 3, 4)
4468905744
4476232672
(1, 2, 3, 4)


**Bug fixed (comment typos):** the original comment said "here id will chnage but change will not happen in tupple" — fixed the typos ("chnage" → "change", "tupple" → "tuple") for readability. The behavior itself was already correct: `L = L + (5, 6)` doesn't mutate the tuple — it creates an **entirely new tuple** and rebinds the *local* name `L` to it, which is why `id(L)` changes inside the function, but the original `L1` outside is untouched. This is the core reason tuples are immutable: there is no in-place "append" operation for them at all.

## 6. 📦 Collections of Objects

Objects can be stored in ordinary Python collections (lists, dicts, etc.) just like any other value.

In [5]:
class Customer:
    def __init__(self, name, age):
        self.name = name
        self.age = age
    
    def intro(self):
        print("I am", self.name,"and my Age is", self.age)

c1 = Customer("Pratham", 23)
c2 = Customer("Isha", 54)
c3 = Customer("Anshu", 73)


L = [c1, c2, c3]

for i in L:
    print(i)

print("---------")

for i in L:
    print(i.name, i.age)
    
print("---------")

for i in L:
    i.intro()


---------
Pratham 23
Isha 54
Anshu 73
---------
I am Pratham and my Age is 23
I am Isha and my Age is 54
I am Anshu and my Age is 73


**Note:** `print(i)` on a raw object shows Python's default representation (`<__main__.Customer object at 0x...>`) because `Customer` doesn't define `__str__` — compare this to the `Fraction` class in Section 2, which *does* define `__str__` and therefore prints nicely.

## 7. 🧮 Static (Class) Variables vs Instance Variables

A **class variable** is shared by every object of the class — perfect for things like counters. An **instance variable** belongs to just one object.

In [ ]:
## Static Variable and Instance Variable

class Customer:
    
    counter = 1
    
    def __init__(self, name, age):
        self.name = name  ## instance variable
        self.age = age ## instance variable
        self.sno = 0
        self.sno += 1
        # print(id(self.sno))
        self.counter = Customer.counter
        Customer.counter += 1
    
    
    def intro(self):
        print("I am", self.name,"and my Age is", self.age)

c1 = Customer("Pratham", 23)
c2 = Customer("Isha", 54)
c3 = Customer("Anshu", 73)

print(c1.sno) ## non static variable
print(c1.sno)
print(c1.sno)
print("----------")
print(c1.counter) ## static variable
print(c2.counter)
print(c3.counter)

1
1
1
----------
1
2
3


**Note:** `self.sno = 0` followed immediately by `self.sno += 1` always resets `sno` to `0` and then increments it to `1` for *every* object — so `c1.sno`, `c2.sno`, and `c3.sno` are all `1`. This variable isn't actually doing what its name ("serial number") suggests; it never persists or increments across objects because it's reset inside `__init__` every time. Compare this to `counter`, which correctly increases (`1`, `2`, `3`) because it reads and writes the shared **class** attribute `Customer.counter` instead of resetting a fresh instance value each time.

### 7.1 A Private Static Counter with a Static Method

In [30]:
## Static Variable and Instance Variable

class Atm:
    
    __counter = 1
    
    def __init__(self, name, age):
        self.name = name  ## instance variable
        self.age = age ## instance variable
        self.__counter = Atm.__counter
        Atm.__counter += 1
       
    @staticmethod
    def get_counter(): ## static method
        return Atm.__counter

    def set_counter(new):
        if type(new) is int:
            Atm.__counter = new
        else:
            print("Not Allowed")
        
    
    def intro(self):
        print("I am", self.name,"and my Age is", self.age)

# c1 = Atm("Pratham", 23)
# c2 = Atm("Isha", 54)

# print(Atm.get_counter())
# c3 = Atm("Anshu", 73)


print(Atm.get_counter())

Atm.set_counter(5)
print(Atm.get_counter())


# print(c1.__counter) ## private static variable cant be accessible use not getter and setter methods
# print(c2.__counter)
# print(c3.__counter)



1
5


**Bug found:** `set_counter(new)` is missing the `@staticmethod` decorator. As written, calling it the way a static method is meant to be called — e.g. `Atm.set_counter(5)` — happens to work only because Python doesn't require an instance when you call a method **through the class itself** (the `new` parameter absorbs the `5`). But calling it on an *instance* (e.g. `some_atm.set_counter(5)`) would crash with `TypeError: set_counter() takes 1 positional argument but 2 were given`, because Python would automatically pass the instance as the first argument, leaving no slot for `5`. Verified below — adding `@staticmethod` fixes it so it works consistently either way.

In [ ]:
## Demonstrating the missing @staticmethod bug, and the fix

class AtmBuggy:
    __counter = 1
    def set_counter(new):   # missing @staticmethod
        Atm_class_ref = new
        return Atm_class_ref

class AtmFixed:
    __counter = 1
    @staticmethod
    def set_counter(new):
        AtmFixed.__counter = new
        return AtmFixed.__counter

buggy = AtmBuggy()
try:
    buggy.set_counter(5)   # crashes: instance auto-passes 'self', leaving no room for 5
except TypeError as e:
    print("AtmBuggy.set_counter on an instance ->", e)

fixed = AtmFixed()
print("AtmFixed.set_counter on an instance ->", fixed.set_counter(5))  # works fine


**Note:** This confirms the fix — with `@staticmethod` added, `set_counter` can be called consistently through the class (`Atm.set_counter(5)`) *or* through an instance (`some_atm.set_counter(5)`) without Python trying to auto-inject `self`.

## 8. 🧩 Class Relationships — Aggregation ("Has-A")

Aggregation means one class holds an object of another class as an attribute. Here, a `Customer` **has an** `Address`.

In [ ]:
class Customer:
    
    def __init__(self, name, gender, address):
        self.name = name
        self.gender = gender
        self.address = address
    
    def edit_profile(self,new_name, new_city, new_pin, new_state):
        self.name = new_name
        self.address.change_address(new_city, new_pin, new_state)

class Address:
    
    def __init__(self, city, pincode, state):
        self.city = city
        self.pincode = pincode
        self.state = state
        
    def change_address(self, new_city, new_pin, new_state):
        self.city = new_city
        self.pincode = new_pin
        self.state = new_state
        
        
add = Address("Bengaluru",829122, "KA")
cust = Customer("Pratham", "Male", add)

print(cust.address)
print(cust.address.city)

cust.edit_profile("Ankit","Gurgano", 12001, "HR")

print(cust.address)
print(cust.address.city)        

Bengaluru
Gurgano


**Note:** `cust.edit_profile(...)` doesn't touch `self.address` directly — it delegates to `self.address.change_address(...)`, letting the `Address` object manage its own fields. This is the payoff of aggregation: each class stays responsible for its own data.

**Note:** `print(cust.address)` shows the default object representation (`<__main__.Address object at 0x...>`) both before and after editing, and the `id()` doesn't change — because `edit_profile` mutates the *existing* `Address` object's attributes rather than replacing it with a new one.

## 9. 🌳 Class Relationships — Inheritance ("Is-A")

Inheritance means a subclass automatically gets all the methods (and attributes) of its parent class. Here, a `Student` **is a** `User`.

In [38]:
class User:
    
    def login(self):
        print("Login")

    def register(self):
        print("register")
    
class Student(User):
    
    def enroll(self):
        print("Enroll")
        
    def review(self):
        print("Review")
    
stu1 = Student()

stu1.enroll()
stu1.review()
stu1.login()
stu1.register()


Enroll
Review
Login
register


**Note:** `Student` doesn't define `login()` or `register()` itself — it inherits them from `User` automatically just by writing `class Student(User):`. This is the core benefit of inheritance: shared behavior is written once in the parent and reused everywhere.

## ⚠ Common Misconceptions

❌ `__variable` in Python makes an attribute truly private, like Java's `private`.
✅ It only triggers **name mangling** (`_ClassName__variable`) — the data is still reachable if you know the mangled name. Python's model is "we're all consenting adults," relying on convention, not enforcement.

❌ `_variable` and `__variable` do the same thing.
✅ `_variable` is a pure convention (nothing happens). `__variable` actually gets renamed by the interpreter.

❌ `@property` getters/setters exist to make attributes "unreadable" from outside.
✅ Their real purpose is to run **validation or extra logic** on read/write while keeping the clean `obj.attr` syntax — not to block access.

❌ Passing an object into a function always protects the original from changes.
✅ Objects are passed by reference — mutating the object's attributes inside the function **does** affect the original. Only *reassigning* the local parameter name (or returning a new object without saving it back) leaves the original untouched.

❌ A missing `@staticmethod` decorator only matters stylistically.
✅ It can cause a real runtime crash when the method is called on an instance instead of the class, because Python then tries to auto-pass `self` as an extra argument.

## 🔍 Interview Questions

- Why doesn't Python have true private variables like Java or C++?
- What is name mangling, and what problem does it actually solve (hint: it's mostly about subclass name clashes, not security)?
- What's the difference between a traditional getter/setter method and a Python `@property`?
- If you pass an object into a function and mutate one of its attributes, does the caller see the change? What if you reassign the parameter to a brand-new object instead?
- What's the difference between a class variable and an instance variable? What happens if an `__init__` accidentally resets what should be a persistent counter?
- Why would forgetting `@staticmethod` cause a `TypeError` only when the method is called on an instance, but not when called on the class?
- What is the difference between Aggregation ("Has-A") and Inheritance ("Is-A")? Give an example of each.

## 🎯 Key Takeaways

1. Python privacy is convention-based: `_x` means "please don't touch," `__x` triggers name mangling (`_ClassName__x`) — neither is a hard security boundary.
2. `@property` + `@x.setter` is the idiomatic way to add validation while keeping simple attribute-style syntax (`obj.x` / `obj.x = value`).
3. Objects (like lists, dicts, sets) are mutable and passed by reference — mutating an attribute inside a function is visible outside it, but reassigning the parameter or returning a new object is not.
4. Class variables are shared across all instances; resetting a value inside `__init__` (like `self.sno = 0`) instead of reading the class-level counter defeats the purpose of a persistent counter.
5. A missing `@staticmethod` can silently "work" when called via the class but crash with `TypeError` when called via an instance — always add it explicitly if `self` isn't needed.
6. Operator-overload methods (`__add__`, `__mul__`, etc.) should return a new instance of the *same class*, not a plain value like a string — otherwise chained operations break.
7. Aggregation ("Has-A": a class holds another as an attribute) and Inheritance ("Is-A": a subclass extends a parent) are the two fundamental ways classes relate to each other.